# 3일차 실습 — YOLO11 로 객체 탐지

부경대학교 교내 컴퓨터비전 부트캠프 · 2026. 8. 5. · 4차시 (12:30 – 13:20)

---

오늘 오전에 배운 **박스 · 클래스 · confidence · IoU · NMS** 가 코드에서 어떻게 나타나는지
직접 확인합니다. 모델을 학습시키지는 않습니다 — COCO 로 사전학습된 YOLO11 을 그대로 씁니다.

| STEP | 하는 일 |
|---|---|
| 0 | 환경 준비 |
| 1 | 첫 추론 — 이미지 한 장 |
| 2 | 결과 뜯어보기 (**TODO 1**) |
| 3 | 직접 시각화 (**TODO 2**) |
| 4 | threshold 실험 (**TODO 3 · 4**) |
| 5 | 내 이미지 · 영상으로 |

**시작 전에 반드시** — 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 선택하세요.

## STEP 0 · 환경 준비

GPU 가 붙어 있는지 먼저 확인합니다. 아래 셀에서 `Tesla T4` 같은 이름이 나와야 합니다.
`command not found` 가 나오면 런타임 유형이 CPU 입니다.

In [ ]:
!nvidia-smi

`ultralytics` 한 패키지에 YOLO11 과 내일 쓸 SAM 이 모두 들어 있습니다. 30초 정도 걸립니다.

In [ ]:
!pip install -q ultralytics

그래프 라벨에 한글이 깨지지 않도록 폰트를 설치합니다. 실패해도 실습에는 지장이 없습니다.

In [ ]:
# 한글 폰트 (약 20초) — 실패해도 그냥 넘어갑니다
try:
    !apt-get install -qq -y fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 준비 완료")
except Exception as e:
    print("폰트 설치 건너뜀:", e)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from ultralytics import YOLO
from ultralytics.utils import ASSETS      # 패키지에 들어 있는 샘플 이미지 폴더

plt.rcParams["figure.dpi"] = 110


def show(img_bgr, title=None, w=11):
    """OpenCV(BGR) 이미지를 노트북에 띄운다. RGB 로 뒤집는 것을 잊지 말 것."""
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1])        # BGR -> RGB
    plt.axis("off")
    if title:
        plt.title(title, fontsize=13)
    plt.show()


print("준비 완료")

## STEP 1 · 첫 추론

`YOLO("yolo11n.pt")` 한 줄이면 모델이 준비됩니다. 가중치 파일이 없으면 자동으로 내려받습니다
(약 6 MB).

- `n` 은 nano — 가장 작고 빠른 크기입니다. 뒤에 `s`, `m`, `l`, `x` 로 갈수록 정확하지만 느립니다.
- `.pt` 는 PyTorch 가중치 파일입니다.

In [ ]:
model = YOLO("yolo11n.pt")

print("클래스 개수:", len(model.names))
print("앞의 10개  :", [model.names[i] for i in range(10)])

COCO 데이터셋의 80개 클래스입니다. **이 목록에 없는 물체는 탐지되지 않습니다** —
STEP 5 에서 직접 확인하게 됩니다.

이제 샘플 이미지 한 장으로 추론합니다.

In [ ]:
IMG = str(ASSETS / "bus.jpg")          # 패키지에 들어 있는 샘플 이미지

results = model(IMG)                   # 추론 — 리스트가 돌아온다
r = results[0]                         # 이미지 1장 -> 결과 1개

show(r.plot(), "r.plot() — ultralytics 가 그려 준 결과")

위 셀의 출력에 `1 bus, 4 persons, 45.2ms` 같은 줄이 찍혔을 것입니다.
`r.plot()` 은 박스와 라벨을 그린 **BGR 넘파이 배열**을 돌려줍니다.

> Colab 에서는 `from google.colab.patches import cv2_imshow` 로 불러온 `cv2_imshow(r.plot())`
> 도 같은 일을 합니다. 여기서는 어디서나 돌아가도록 matplotlib 을 씁니다.

## STEP 2 · 결과 뜯어보기

그림이 아니라 **숫자**를 봅니다. 오전에 배운 세 가지가 그대로 들어 있습니다.

| 속성 | 모양 | 내용 |
|---|---|---|
| `r.boxes.xyxy` | (N, 4) | 픽셀 좌표 x1, y1, x2, y2 |
| `r.boxes.conf` | (N,) | confidence |
| `r.boxes.cls` | (N,) | 클래스 번호 (실수형) |
| `r.names` | dict | 번호 → 이름 |

In [ ]:
print("탐지된 객체 수:", len(r.boxes))
print()
print("xyxy\n", r.boxes.xyxy)
print()
print("conf :", r.boxes.conf)
print("cls  :", r.boxes.cls)

### TODO 1

`r.boxes` 를 순회하며 객체마다 **이름 · confidence · 좌표** 를 한 줄씩 출력하세요.

출력 예시

```
person       0.89   (671, 395) - (810, 879)
bus          0.94   (  4, 229) - (796, 728)
```

힌트
- `for box in r.boxes:` 로 하나씩 꺼낼 수 있습니다.
- `box.xyxy` 는 `(1, 4)` 모양입니다 — `box.xyxy[0]` 로 4개 값을 꺼냅니다.
- `box.cls` 는 실수형이라 `int()` 로 바꿔야 `r.names` 의 키로 쓸 수 있습니다.

In [ ]:
for box in r.boxes:
    # TODO 1 ── 아래 네 줄을 채우세요
    x1, y1, x2, y2 = ...          # box.xyxy 에서 좌표 4개 꺼내기
    name = ...                    # 클래스 번호 -> 이름
    conf = ...                    # confidence 를 float 으로
    print(f"{name:<12} {conf:.2f}   ({x1:4.0f}, {y1:4.0f}) - ({x2:4.0f}, {y2:4.0f})")

클래스별로 몇 개가 나왔는지도 세어 봅니다. (이 셀은 그대로 실행하세요)

In [ ]:
names = [r.names[int(c)] for c in r.boxes.cls]
for name, n in Counter(names).most_common():
    print(f"{name:<12} {n}개")

### 잠깐 — 좌표를 그림 위에서 확인하기

`bus.jpg` 는 810 × 1080 픽셀입니다. 위에서 나온 `person` 의 x 좌표가 671 ~ 810 이라면
**이미지의 오른쪽 끝**에 있는 사람입니다. 좌표계의 원점은 **왼쪽 위**라는 것을 기억하세요.

In [ ]:
h, w = cv2.imread(IMG).shape[:2]
print(f"이미지 크기 : 가로 {w}, 세로 {h}")
print(f"박스 좌표는 이 범위 안의 픽셀 값입니다 — x 는 0~{w}, y 는 0~{h}")

## STEP 3 · 직접 시각화

`r.plot()` 은 편하지만 색과 글씨를 바꿀 수 없습니다. 직접 그려 보면 좌표계가 몸에 익습니다.

주의할 점
- `cv2.rectangle` 은 좌표를 **정수**로 받습니다 — `map(int, ...)` 로 변환해야 합니다.
- OpenCV 의 색은 **BGR** 순서입니다. `(255, 128, 0)` 은 빨강이 아니라 파랑 계열입니다.

In [ ]:
# 클래스마다 다른 색을 주기 위한 팔레트 (BGR)
PALETTE = [(232, 96, 21), (50, 113, 233), (91, 125, 46),
           (163, 79, 122), (60, 60, 220), (200, 160, 40)]


def color_of(cls_id):
    return PALETTE[int(cls_id) % len(PALETTE)]


print("팔레트 준비 완료")

### TODO 2

`draw_boxes()` 안의 두 줄을 채워 박스와 라벨을 그리세요.

- `cv2.rectangle(img, (x1, y1), (x2, y2), color, 두께)`
- `cv2.putText(img, 글자, (x, y), 폰트, 크기, color, 두께)`
  - 폰트는 `cv2.FONT_HERSHEY_SIMPLEX` 를 쓰면 됩니다.
  - 라벨은 박스 **위쪽**에 놓아야 읽기 좋습니다 — y 좌표를 조금 빼세요.

In [ ]:
def draw_boxes(img_path, result, thickness=2):
    img = cv2.imread(img_path)
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        color = color_of(box.cls)
        label = f"{result.names[int(box.cls)]} {float(box.conf):.2f}"

        # TODO 2 ── 아래 두 줄을 채우세요
        ...          # 박스 그리기   (cv2.rectangle)
        ...          # 라벨 쓰기     (cv2.putText, 박스 위쪽에)
    return img


show(draw_boxes(IMG, r), "직접 그린 결과")

잘 그려졌다면 `r.plot()` 결과와 거의 같은 그림이 나옵니다.

색이 이상해 보인다면 — `show()` 안에서 `[:, :, ::-1]` 로 BGR 을 RGB 로 뒤집고 있다는 점을
떠올려 보세요. `cv2.imread` 로 읽고 matplotlib 으로 띄울 때 늘 붙는 변환입니다.

## STEP 4 · threshold 실험

오전 2차시의 **두 개의 손잡이**를 직접 돌려 봅니다.

| 인자 | 무엇을 정하나 | 기본값 |
|---|---|---|
| `conf` | 몇 점부터 예측으로 인정할지 | 0.25 |
| `iou` | NMS 에서 얼마나 겹쳐야 중복으로 볼지 | 0.7 |

먼저 사람이 여러 명 겹쳐 있는 이미지로 바꿉니다 — 차이가 더 잘 보입니다.

In [ ]:
IMG2 = str(ASSETS / "zidane.jpg")
show(cv2.imread(IMG2), "실험에 쓸 이미지")

### TODO 3 — conf 를 바꿔 본다

`conf` 를 `0.05, 0.10, 0.25, 0.50, 0.70` 으로 바꿔 가며 탐지 개수를 비교하세요.

힌트 — `model(IMG2, conf=c, verbose=False)` 로 인자를 넘깁니다. 결과 개수는 `len(res[0].boxes)`.

In [ ]:
conf_list = [0.05, 0.10, 0.25, 0.50, 0.70]
counts = []

for c in conf_list:
    # TODO 3 ── 두 줄을 채우세요
    res = ...                    # conf=c 로 추론
    n = ...                      # 탐지된 박스 개수
    counts.append(n)
    print(f"conf = {c:.2f}  ->  {n:2d}개")

숫자만으로는 감이 안 오니 그림으로도 봅니다. (그대로 실행)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, c in zip(axes, [0.05, 0.25, 0.70]):
    res = model(IMG2, conf=c, verbose=False)[0]
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"conf = {c}   ({len(res.boxes)}개)", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

**관찰해 보세요**

- `conf` 를 낮추면 박스가 늘어납니다. 늘어난 박스들이 실제로 물체인가요?
- `conf=0.70` 에서 사라진 박스 중에 진짜 물체가 있나요?

이것이 오전에 본 **Precision 과 Recall 의 맞바꿈**입니다.

### TODO 4 — iou 를 바꿔 본다

이번에는 NMS 의 `iou` 기준을 바꿉니다. `conf` 는 0.10 으로 고정해 박스를 넉넉히 남겨 둡니다.

`iou` 를 `0.3, 0.7, 0.9` 로 바꿔 가며 개수를 출력하고, 그림도 함께 그려 보세요.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, v in zip(axes, [0.3, 0.7, 0.9]):
    # TODO 4 ── 한 줄을 채우세요 (conf 는 0.10 으로 고정, iou 만 v 로)
    res = ...
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"iou = {v}   ({len(res.boxes)}개)", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

**관찰해 보세요**

- `iou=0.3` — 조금만 겹쳐도 지웁니다. 나란히 선 두 사람이 하나로 합쳐지지 않았나요?
- `iou=0.9` — 거의 안 지웁니다. 같은 사람에게 박스가 여러 개 붙지 않았나요?

기본값 0.7 이 왜 무난한 선택인지 눈으로 확인되면 이 STEP 은 끝입니다.

### 그 밖의 유용한 인자 (그대로 실행)

`classes` 로 관심 있는 클래스만 남기면 결과가 훨씬 읽기 쉬워집니다.
COCO 기준 `0 = person`, `2 = car`, `9 = traffic light` 입니다.

In [ ]:
res = model(IMG, classes=[0], conf=0.25, verbose=False)[0]   # 사람만
print("사람만:", len(res.boxes), "개")
show(res.plot(), "classes=[0] — 사람만 남긴 결과")

아까 전체 클래스로 돌렸을 때와 사람 개수가 하나 다를 수 있습니다.
`classes` 필터는 NMS **이전에** 적용되기 때문에, 다른 클래스 박스와의 경쟁이 사라지면서
살아남는 박스가 달라질 수 있습니다. 버그가 아니라 후처리 순서 때문입니다.

## STEP 5 · 내 이미지 · 영상으로

**여기가 오늘 실습에서 가장 중요한 부분입니다.** 여기서 쓴 이미지를 내일 SAM 실습에서
그대로 다시 씁니다.

아래 셀을 실행하면 파일 선택 창이 뜹니다. 사진을 1 ~ 3장 올려 보세요.
사람 · 자동차 · 강아지 · 노트북 · 의자처럼 COCO 에 있는 물체가 담긴 사진이 좋습니다.

In [ ]:
# Colab 전용 — 로컬에서 실행 중이라면 이 셀은 건너뛰고 my_images 에 경로를 직접 적으세요
try:
    from google.colab import files
    uploaded = files.upload()
    my_images = list(uploaded.keys())
except Exception as e:
    print("Colab 이 아닙니다:", e)
    my_images = []

print("업로드된 파일:", my_images)

In [ ]:
# 업로드한 이미지가 없으면 샘플로 대체합니다
my_images = globals().get("my_images", [])
if not my_images:
    my_images = [str(ASSETS / "bus.jpg"), str(ASSETS / "zidane.jpg")]

for path in my_images:
    res = model(path, conf=0.25, verbose=False)[0]
    found = Counter(res.names[int(c)] for c in res.boxes.cls)
    print(path, "->", dict(found) if found else "탐지된 객체 없음")
    show(res.plot(), path)

### 확인해 볼 것

1. **COCO 에 없는 물체**가 사진에 있다면 어떻게 되었나요?
   보통 아무것도 나오지 않거나, 가장 비슷하게 생긴 다른 클래스로 잘못 붙습니다.
2. **작은 물체**는 잘 잡히나요? 멀리 있는 사람이나 표지판을 확인해 보세요.
   안 잡힌다면 `imgsz=1280` 으로 해상도를 올려 다시 시도해 보세요.
3. **가려진 물체**는 어떤가요? 절반쯤 가려진 물체에 붙은 confidence 를 살펴보세요.

In [ ]:
# 해상도를 올려 작은 물체를 다시 시도 (기본 640 -> 1280)
path = my_images[0]
for size in [640, 1280]:
    res = model(path, imgsz=size, conf=0.25, verbose=False)[0]
    print(f"imgsz={size:4d}  ->  {len(res.boxes)}개")

### 영상으로도 (선택 — 시간이 남으면)

프레임이 많은 영상은 반드시 `stream=True` 를 붙여야 합니다.
안 붙이면 모든 프레임의 결과를 메모리에 쌓다가 런타임이 죽습니다.

```python
for res in model("my_video.mp4", stream=True, conf=0.25):
    print(len(res.boxes), end=" ")
```

결과 영상을 파일로 저장하려면 `save=True` 를 붙입니다 — `runs/detect/predict/` 아래에 생깁니다.

## 정리

오늘 확인한 것을 한 문장씩 적어 보세요. (이 셀을 더블클릭해 직접 수정)

1. `conf` 를 낮추면 …
2. `iou` 를 낮추면 …
3. 내 사진에서 잘 안 잡힌 물체는 … 이고, 이유는 …

---

### 내일 (8/6) 예고

박스가 아니라 **픽셀 단위 마스크**를 다룹니다. 오늘 만든 탐지 박스를 그대로 SAM 에
prompt 로 넘기면 그 자리의 마스크가 나옵니다 — 최종 프로젝트 주제 1번이 정확히 이 조합입니다.

STEP 5 에서 업로드한 이미지는 지우지 말고 가지고 계세요.